# Chapter 7: LLM Integration

## Using Claude to Do the Boring Parts of ML Ops

Our pipeline has two places where a task is fundamentally about *understanding language* — something humans do effortlessly but rule-based code struggles with:

1. **Auto-Mapping (onboarding):** A new client uploads a CSV. Someone has to figure out what `MonthlyCharges`, `mrr`, or `amt_per_month` all mean. That's pattern recognition on natural language.

2. **Narrative Generation (output):** SHAP gives us `contract_type=month-to-month (+0.23)`. A business user needs: "This customer has no long-term commitment." That's translation from numbers to English.

Both tasks use Amazon Bedrock (Claude) via API. Both are **non-blocking** — if Bedrock fails, the pipeline continues without these features. No data is lost.

### Cost

| Task | Typical Cost | When It Runs |
|------|-------------|-------------|
| Auto-Mapping | ~$0.01 per new client | Once, during onboarding |
| Narratives (50 customers) | ~$0.01-0.02 per batch | Each scoring run |

In [ ]:
from churn_pipeline.llm.auto_mapping import (
    build_mapping_prompt,
    write_draft_yaml,
    is_mapping_approved,
    ColumnMapping,
    _parse_mapping_response,
)
from churn_pipeline.llm.narrative_generator import (
    build_narrative_prompt,
    parse_narrative_response,
    generate_narratives_for_batch,
    NarrativeRequest,
    SYSTEM_PROMPT,
)
from churn_pipeline.data_contract import STANDARD_SCHEMA

## Part 1: Auto-Mapping — LLM Reads Your Column Names

### The Problem

Every company calls their data something different:

| Company A | Company B | Company C | They All Mean... |
|-----------|-----------|-----------|------------------|
| MonthlyCharges | mrr | monthly_fee | monthly_charges |
| customerID | CustID | user_id | customer_id |
| Churn | left_service | is_churned | churn_label |

A rule-based approach would need an infinite dictionary of synonyms. An LLM handles this naturally because it *understands language*.

In [ ]:
# Simulate what a new client's CSV looks like
client_columns = ["CustID", "months_active", "MonthlyFee", "TotalSpend",
                   "left_service", "plan_type", "how_they_pay", "complaints"]

sample_rows = [
    {"CustID": "USR-7590", "months_active": 12, "MonthlyFee": 59.99,
     "TotalSpend": 719.88, "left_service": "no", "plan_type": "annual",
     "how_they_pay": "credit card", "complaints": 0},
    {"CustID": "USR-3344", "months_active": 2, "MonthlyFee": 99.99,
     "TotalSpend": 199.98, "left_service": "yes", "plan_type": "monthly",
     "how_they_pay": "bank transfer", "complaints": 4},
]

print("Client's raw columns:", client_columns)
print(f"\nSample row: {sample_rows[0]}")

In [ ]:
# Build the prompt we'd send to Claude
prompt = build_mapping_prompt(client_columns, sample_rows)

print("Prompt sent to Claude (first 800 chars):")
print("=" * 60)
print(prompt[:800])
print("...")
print(f"\n(Total prompt length: {len(prompt)} characters)")

In [ ]:
# Simulate what Claude would respond with
# (In production, this comes from Bedrock API)
simulated_response = """[
    {"source_column": "CustID", "target_field": "customer_id", "confidence": "high", "reasoning": "Contains 'ID' and values look like unique identifiers"},
    {"source_column": "months_active", "target_field": "tenure_months", "confidence": "high", "reasoning": "Directly describes duration in months"},
    {"source_column": "MonthlyFee", "target_field": "monthly_charges", "confidence": "high", "reasoning": "Monthly + Fee = monthly billing amount"},
    {"source_column": "TotalSpend", "target_field": "total_charges", "confidence": "high", "reasoning": "Cumulative spending"},
    {"source_column": "left_service", "target_field": "churn_label", "confidence": "medium", "reasoning": "Binary indicator of leaving, needs value mapping yes/no to 1/0"},
    {"source_column": "plan_type", "target_field": "contract_type", "confidence": "medium", "reasoning": "Describes contract duration category"},
    {"source_column": "how_they_pay", "target_field": "payment_method", "confidence": "high", "reasoning": "Payment method description"},
    {"source_column": "complaints", "target_field": "support_tickets", "confidence": "medium", "reasoning": "Complaint count likely correlates with support interactions"}
]"""

# Parse the response
mappings = _parse_mapping_response(simulated_response)

print("Claude's mapping suggestions:")
print("=" * 70)
print(f"{'Client Column':<18} {'→ Standard Field':<20} {'Confidence':<12} Reasoning")
print("-" * 70)
for m in mappings:
    print(f"{m.source_column:<18} → {m.target_field:<18} {m.confidence:<12} {m.reasoning[:40]}")

### The Approval Workflow

The LLM's output is a **draft** — it writes a `.draft.yaml` file that a human must review:

1. LLM generates `mapping.draft.yaml` with confidence scores
2. Human reviews: fixes mistakes, adds value_mappings ("yes"→1, "no"→0)
3. Human renames to `mapping.yaml` → pipeline now trusts it

The pipeline NEVER uses a draft directly. This human-in-the-loop step prevents bad mappings from corrupting data.

In [ ]:
import tempfile, os

# Write the draft
tmp_dir = tempfile.mkdtemp()
draft_path = os.path.join(tmp_dir, "mapping.draft.yaml")
approved_path = os.path.join(tmp_dir, "mapping.yaml")

write_draft_yaml(mappings, "new_client", draft_path)

print("Draft written. Approval status:")
print(f"  mapping.draft.yaml exists: {os.path.exists(draft_path)}")
print(f"  is_mapping_approved(draft): {is_mapping_approved(draft_path)}")
print(f"  is_mapping_approved(mapping.yaml): {is_mapping_approved(approved_path)}")

# Simulate human approval (rename)
os.rename(draft_path, approved_path)
print(f"\nAfter human renames file:")
print(f"  is_mapping_approved(mapping.yaml): {is_mapping_approved(approved_path)} ✓")

# Cleanup
os.unlink(approved_path)
os.rmdir(tmp_dir)

## Part 2: Narrative Generation — SHAP Numbers to English

### The Problem

After scoring, we have:
```
contract_type=month-to-month (+0.23); tenure_months=2 (+0.15); support_tickets=5 (+0.18)
```

A data scientist reads that and understands. A VP of Customer Success needs:

> "This customer has no long-term commitment and has contacted support 5 times in just 2 months. Month-to-month customers with high support volume are among the most likely to leave. Consider offering a discounted annual plan."

The LLM does this translation at scale.

In [ ]:
# Build a batch of customers needing narratives
batch = [
    NarrativeRequest(
        customer_id="CUST_001",
        churn_probability=0.87,
        risk_tier="high",
        top_shap_features=[
            {"feature": "contract_type", "contribution": 0.23},
            {"feature": "support_tickets", "contribution": 0.18},
            {"feature": "tenure_months", "contribution": 0.15},
        ],
    ),
    NarrativeRequest(
        customer_id="CUST_002",
        churn_probability=0.72,
        risk_tier="high",
        top_shap_features=[
            {"feature": "monthly_charges", "contribution": 0.19},
            {"feature": "contract_type", "contribution": 0.16},
            {"feature": "tenure_months", "contribution": 0.12},
        ],
    ),
]

feature_defs = {
    "contract_type": "Whether the customer is on month-to-month, annual, or two-year plan",
    "support_tickets": "Number of times the customer contacted support",
    "tenure_months": "How long they've been a customer",
    "monthly_charges": "What they pay each month",
}

prompt = build_narrative_prompt(batch, feature_definitions=feature_defs)

print("Narrative prompt (sent to Claude):")
print("=" * 60)
print(prompt[:1200])
print("...")

In [ ]:
# Simulate Claude's response
simulated_narrative_response = """CUSTOMER_ID: CUST_001
NARRATIVE: This customer is at very high risk of leaving. They have no long-term commitment (month-to-month plan), which means there's zero friction to cancel. They've also contacted support 5 times in just 2 months — a strong signal of frustration. With only 2 months of tenure, they haven't built any loyalty yet. Consider offering a discounted annual plan to lock them in, and escalate their open support issues immediately.

CUSTOMER_ID: CUST_002
NARRATIVE: This customer is paying significantly more than average ($110/month) on a month-to-month plan. High charges without a commitment create a "why am I paying this much?" moment. They're relatively new (4 months), so they're still in the window where switching costs are low. A loyalty discount or plan review could reduce their perceived cost and extend their stay."""

# Parse the response
narratives = parse_narrative_response(simulated_narrative_response, ["CUST_001", "CUST_002"])

print("Generated narratives:")
print("=" * 60)
for cust_id, narrative in narratives.items():
    print(f"\n{cust_id}:")
    print(f"  {narrative[:200]}..." if len(narrative) > 200 else f"  {narrative}")

## Failure Handling: Non-Blocking by Design

Both LLM steps are designed to fail gracefully:

| Failure | What Happens | Client Impact |
|---------|-------------|---------------|
| Auto-mapping fails | Operator writes YAML manually | Slower onboarding, same result |
| Narrative generation fails | narrative_explanation = "N/A" | Client still gets SHAP reasons |

The pipeline NEVER crashes because of an LLM failure. These are enhancements, not dependencies.

In [ ]:
# Demonstrate failure handling
results = generate_narratives_for_batch(
    batch,
    boto3_client="intentionally_invalid",  # Force failure
)

print("When Bedrock fails:")
print("=" * 40)
for cust_id, result in results.items():
    print(f"  {cust_id}: narrative='{result.narrative}', success={result.success}")

print("\n→ Pipeline continues. Client gets SHAP reasons in top_3_reasons column.")
print("→ narrative_explanation says 'N/A' instead of English paragraph.")
print("→ No crash. No data loss. Just slightly less readable output.")

## The System Prompt: Controlling Claude's Output

The system prompt sets boundaries for what Claude writes. Key constraints:

- **Non-technical language** — no "SHAP values" or "feature importance"
- **Under 150 words** — concise, actionable
- **Reference specific values** — "5 support tickets" not "high support volume"
- **Plain English** — a VP should understand every word

In [ ]:
print("System prompt used for narrative generation:")
print("=" * 60)
print(SYSTEM_PROMPT)

## Key Takeaways

1. **Two LLM steps:** auto-mapping (onboarding) + narratives (output) — both are language problems
2. **Non-blocking:** if Bedrock fails, the pipeline continues without these features
3. **Human-in-the-loop:** auto-mapping produces a DRAFT that requires human approval
4. **Batch processing:** 50 customers in one prompt, not 50 API calls
5. **Cost:** pennies per run, not dollars
6. **System prompt constraints:** under 150 words, non-technical, reference specific values

Next: Chapter 8 covers the AWS architecture — how SageMaker Pipelines orchestrates all these components into a production system.

---

*Source code: `src/churn_pipeline/llm/auto_mapping.py`, `src/churn_pipeline/llm/narrative_generator.py`*  
*Tests: `tests/unit/test_auto_mapping.py`, `tests/property/test_narrative.py`, `tests/unit/test_narrative.py`*  
*Series: [Build with AWS](https://buildwithaws.substack.com/)*